<style>
/* Three-layer translation blocks */
.three-layer {
    display: grid;
    grid-template-columns: 1fr 1fr 1fr;
    gap: 10px;
    margin: 14px 0;
    font-size: 0.95em;
}
.three-layer .panel {
    border-radius: 8px;
    padding: 10px 14px;
    border: 1px solid transparent;
}
.three-layer .panel h4 {
    margin: 0 0 8px 0;
    font-size: 0.75em;
    text-transform: uppercase;
    letter-spacing: 0.08em;
    font-weight: 700;
    padding-bottom: 4px;
    border-bottom: 1px solid currentColor;
}
.three-layer .panel-math {
    background: #eef2f8;
    color: #1e3a5f;
    border-color: #c7d2e0;
}
.three-layer .panel-plain {
    background: #eef4ef;
    color: #2d5f4e;
    border-color: #c8d9ce;
}
.three-layer .panel-code {
    background: #f7efe0;
    color: #6b3e0f;
    border-color: #e0c8a0;
    font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
    font-size: 0.85em;
}
.three-layer .panel pre, .three-layer .panel code {
    background: transparent;
    color: inherit;
    padding: 0;
}
@media (max-width: 1000px) {
    .three-layer { grid-template-columns: 1fr; }
}

/* Callouts */
.callout {
    border-left: 4px solid;
    padding: 10px 16px;
    margin: 16px 0;
    border-radius: 0 6px 6px 0;
}
.callout h4 {
    margin: 0 0 6px 0;
    text-transform: uppercase;
    font-size: 0.72em;
    letter-spacing: 0.08em;
    font-weight: 700;
}
.callout-definition  { border-color: #1e3a5f; background: #f5f8fc; }
.callout-theorem     { border-color: #2d5f4e; background: #f3f8f5; }
.callout-exercise    { border-color: #8a6d1e; background: #faf6eb; }
.callout-insight     { border-color: #7c3a5e; background: #faf0f5; }
.callout-codebase    { border-color: #333; background: #f6f6f4; }
.callout-warning     { border-color: #a04020; background: #faf0ec; }

/* Exercise solutions (collapsible) */
details.solution {
    margin-top: 8px;
    padding: 8px 12px;
    background: #fdfcf7;
    border: 1px dashed #b8a470;
    border-radius: 4px;
}
details.solution summary {
    cursor: pointer;
    font-size: 0.85em;
    text-transform: uppercase;
    letter-spacing: 0.06em;
    font-weight: 700;
    color: #6b5420;
}

/* Module nav */
.module-nav {
    display: flex;
    justify-content: space-between;
    font-size: 0.9em;
    color: #555;
    margin: 24px 0 8px 0;
    padding: 8px 0;
    border-top: 1px solid #ddd;
}
</style>

# Module 2 — Backward Stochastic Differential Equations

> *The object the repo exists to solve. We go backward from a known terminal value and discover the running cost $f$ and the "hedging" process $Z$.*

## Learning goals

1. **State a BSDE** in the canonical form $dY_t = -f(t, X_t, Y_t, Z_t)\,dt + Z_t\,dW_t$ with $Y_T = g(X_T)$, and explain the role of every symbol.
2. **Use the martingale representation theorem** to justify why $Z_t$ exists and is unique.
3. **Interpret $Y_t$ and $Z_t$** as value and sensitivity of a stochastic control problem.
4. **State Pardoux–Peng** (existence, uniqueness under Lipschitz generator).
5. **Derive the semilinear-PDE companion** of a Markovian BSDE (generalised Feynman–Kac).
6. **Read [equations/base.py](../equations/base.py) and [equations/contxiong_lob.py:302-370](../equations/contxiong_lob.py)** and identify the `f_tf` generator and `g_tf` terminal condition.

## Prereqs

Module 1 (Brownian motion, Itô's formula, SDEs, Euler–Maruyama).


## 1. Why go backward?

Suppose you know the final value of something — say, the payoff of an option at expiry — and you want to compute its value *now*. In a deterministic world you could just solve an ODE backward. In a stochastic world you face a subtle problem: the process you construct must **remain adapted** at all intermediate times (no peeking at the future).

The resolution, discovered by Pardoux & Peng (1990), is to write the backward equation in the form

$$Y_t = g(X_T) + \int_t^T f(s, X_s, Y_s, Z_s)\,ds - \int_t^T Z_s\,dW_s.$$

The extra process $Z_t$ is the price you pay for adaptedness: it is the Brownian "hedging" component that keeps $Y_t$ a well-defined conditional expectation at every time.

BSDEs unify three seemingly distinct pieces of machinery:

| Classical object | BSDE form |
|-----------------|-----------|
| European option price $V(t, x) = \mathbb{E}[g(X_T) \mid X_t = x]$ | $f \equiv 0$ |
| Semilinear PDE $\partial_t u + \mathcal{L} u + f(t, x, u, \sigma \nabla u) = 0$ with $u(T, x) = g(x)$ | General $f$ |
| HJB equation of stochastic control | $f$ = running cost minus Hamiltonian |

The repo uses the last one: the value function of a market-maker is a BSDE.


## 2. Definition

<div class="three-layer">
<div class="panel panel-math"><h4>Math</h4>

Let $X_t$ be a forward SDE and $T > 0$ a terminal time. A **BSDE** is a pair $(Y, Z)$ of adapted processes satisfying

$$Y_t = g(X_T) + \int_t^T f(s, X_s, Y_s, Z_s)\,ds - \int_t^T Z_s\,dW_s,$$

or equivalently in differential form,

$$-dY_t = f(t, X_t, Y_t, Z_t)\,dt - Z_t\,dW_t, \qquad Y_T = g(X_T).$$

- $g : \mathbb{R}^d \to \mathbb{R}$ is the **terminal** condition.
- $f : [0,T] \times \mathbb{R}^d \times \mathbb{R} \times \mathbb{R}^d \to \mathbb{R}$ is the **generator** (or driver).
- $Y$ is scalar, $Z$ is $d$-dimensional (matches $W$).

</div>
<div class="panel panel-plain"><h4>Plain English</h4>

Think of $Y_t$ as a *value*: "how much is my portfolio / option / position worth at time $t$, given what I know so far?"

$Y_T = g(X_T)$ pins down the terminal value — e.g., the payoff at expiry, or a liquidation penalty on leftover inventory.

The generator $f$ is a **running cost** (or drift) that $Y$ accumulates going backward in time.

$Z_t$ is the slope of $Y$ with respect to Brownian shocks. Economically, it is the number of units of the risky asset you need to *hold* at time $t$ to hedge the fluctuation of $Y$ from time $t$ to $t + dt$. It is uniquely determined by the martingale representation theorem (next section).

</div>
<div class="panel panel-code"><h4>Code</h4>

The abstract interface lives in
[equations/base.py](../equations/base.py):

<pre>class Equation:
    def sample(self, n):
        # returns X paths
    def f_tf(self, t, x, y, z):
        # the generator
    def g_tf(self, t, x):
        # the terminal</pre>

Every concrete equation in
`equations/contxiong_lob*.py`
just fills in these three
methods.

</div>
</div>

## 3. The martingale representation theorem

Where does $Z_t$ come from? It is forced on us by the following fundamental fact.


<div class="callout callout-theorem"><h4>Martingale Representation (for BM filtration)</h4>

Let $(\mathcal{F}_t)$ be the filtration generated by a $d$-dimensional Brownian motion $W$. Then every square-integrable $\mathcal{F}_T$-measurable random variable $\xi$ admits a unique representation

$$\xi = \mathbb{E}[\xi] + \int_0^T Z_s\,dW_s$$

for some adapted, square-integrable process $Z$. In particular, the martingale $M_t = \mathbb{E}[\xi \mid \mathcal{F}_t]$ is a stochastic integral:

$$M_t = \mathbb{E}[\xi] + \int_0^t Z_s\,dW_s.$$

</div>

**Consequence for BSDEs.** If $f \equiv 0$, then $Y_t = \mathbb{E}[g(X_T) \mid \mathcal{F}_t]$ is a martingale; by the theorem it equals $\mathbb{E}[g(X_T)] + \int_0^t Z_s\,dW_s$, which gives $-dY_t = -Z_t\,dW_t$. The non-trivial generator $f$ just adds a drift to the dynamics of $Y$.

**Uniqueness of $Z$.** The representation is unique up to $dt \otimes d\mathbb{P}$-null sets. This is what makes BSDEs *well-posed*: for each admissible $Y$, there is exactly one $Z$ that makes the equation work.


## 4. The Y and Z interpretation

A useful mental model: $Y$ is the **value** of a position, and $Z$ is the **delta** (price sensitivity per unit of Brownian shock).


<div class="three-layer">
<div class="panel panel-math"><h4>Math</h4>

**Linear BSDE (closed form).** Take $f(y) = -r y$ (discounting). The BSDE

$$-dY_t = -r Y_t\,dt - Z_t\,dW_t, \quad Y_T = g(X_T)$$

has explicit solution

$$Y_t = \mathbb{E}\bigl[e^{-r(T-t)} g(X_T) \,\bigm|\, \mathcal{F}_t\bigr],$$

and $Z_t$ is the integrand produced by the martingale representation of $e^{-rT} g(X_T)$.

</div>
<div class="panel panel-plain"><h4>Plain English</h4>

When the generator is **linear**, $Y_t$ is a straightforward (risk-neutral) conditional expectation — discounted future payoff.

This is exactly the classical Black–Scholes formula for a vanilla option: $Y_0$ is the option price, $Z_0 / \sigma$ is the delta hedge ratio at $t=0$.

For non-linear $f$ — and the repo has a very non-linear $f$ — you lose this closed form, but the same "value + hedge" picture still holds.

</div>
<div class="panel panel-code"><h4>Code</h4>

In a linear-generator test you
can compute $Y_0$ two ways:

(1) Deep BSDE solver
(2) Monte Carlo: <pre>Y0_mc = np.mean(
   np.exp(-r*T) * g(X_T))</pre>

The solver must match MC. This
is a standard sanity check —
not in the repo's current tests
but a classic smoke test for
any BSDE library.

</div>
</div>

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (9, 3.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

# Linear BSDE: -dY = -r Y dt - Z dW, Y_T = g(X_T)
# Forward: dX = mu X dt + sigma X dW (GBM)
# Payoff: g(x) = max(x - K, 0)   [European call]
# Y_0 = exp(-r T) E[g(X_T)] for this choice.

def simulate_gbm(S0, mu, sigma, T, N, n_paths, seed=0):
    r = np.random.default_rng(seed)
    dt = T / N
    dW = r.standard_normal((N, n_paths)) * np.sqrt(dt)
    logS = np.log(S0) + np.cumsum((mu - 0.5 * sigma**2) * dt + sigma * dW, axis=0)
    return np.concatenate([[np.log(S0) * np.ones(n_paths)], logS], axis=0)

S0, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20
N, n_paths = 200, 50_000

logS = simulate_gbm(S0, r, sigma, T, N, n_paths)
S_T = np.exp(logS[-1])
payoff = np.maximum(S_T - K, 0.0)
Y0_mc = np.exp(-r * T) * payoff.mean()
mc_se = np.exp(-r * T) * payoff.std() / np.sqrt(n_paths)

# Black-Scholes closed form for comparison
from math import log, sqrt, erf
def bs_call(S0, K, T, r, sigma):
    d1 = (log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)
    N = lambda x: 0.5 * (1 + erf(x / sqrt(2)))
    return S0 * N(d1) - K * np.exp(-r * T) * N(d2)

bs_price = bs_call(S0, K, T, r, sigma)

print(f"Linear BSDE with call payoff:")
print(f"  MC estimate of Y_0      = {Y0_mc:.4f}  (+/- {1.96*mc_se:.4f})")
print(f"  Black-Scholes closed    = {bs_price:.4f}")
print(f"  Discrepancy            = {abs(Y0_mc - bs_price):.4f}")


## 5. Existence and uniqueness — Pardoux–Peng

<div class="callout callout-theorem"><h4>Pardoux–Peng (1990)</h4>

Suppose

- $g : \mathbb{R}^d \to \mathbb{R}$ satisfies $\mathbb{E}[|g(X_T)|^2] < \infty$.
- $f(t, x, y, z)$ is jointly Lipschitz in $(y, z)$: there exists $K > 0$ such that
$$|f(t, x, y_1, z_1) - f(t, x, y_2, z_2)| \leq K\bigl(|y_1 - y_2| + \|z_1 - z_2\|\bigr).$$
- $\mathbb{E}\!\left[\int_0^T |f(t, X_t, 0, 0)|^2\,dt\right] < \infty$.

Then the BSDE admits a **unique adapted, square-integrable** solution $(Y, Z)$ on $[0, T]$.

*Proof sketch.* A Banach fixed-point argument: define a map $(y, z) \mapsto (Y, Z)$ where $(Y, Z)$ is the solution of the BSDE with generator frozen at $(y, z)$. Lipschitz + Gronwall gives a contraction in a weighted $L^2$ norm.

</div>

<div class="callout callout-insight"><h4>Why Lipschitz matters for deep BSDE</h4>

The Pardoux–Peng proof extends to the deep-BSDE *numerical* method: when the generator is Lipschitz and the terminal is square-integrable, the Monte-Carlo + neural-network scheme of Han–Jentzen–E (Module 6) is provably consistent. For non-Lipschitz generators (quadratic growth in $z$, etc.), more delicate theory is needed — and in fact the repo's generator *is* Lipschitz in the relevant range because of the clamping in `_exec_prob_tf` at [contxiong_lob.py:115-124](../equations/contxiong_lob.py). That clamp is not just numerical — it buys you Lipschitz constants, hence existence guarantees.

</div>

## 6. BSDEs and semilinear PDEs — generalised Feynman–Kac

<div class="three-layer">
<div class="panel panel-math"><h4>Math</h4>

Assume $X_t$ is Markovian with generator
$\mathcal{L} = \mu(x)\partial_x + \tfrac{1}{2}\sigma(x)\sigma(x)^\top : \partial_{xx}$.

Let $u : [0, T] \times \mathbb{R}^d \to \mathbb{R}$ solve the semilinear PDE

$$\partial_t u + \mathcal{L}u + f(t, x, u, \sigma^\top \nabla u) = 0,$$
$$u(T, x) = g(x).$$

Then $Y_t = u(t, X_t)$ and $Z_t = \sigma(t, X_t)^\top \nabla u(t, X_t)$ solve the Markovian BSDE with generator $f$ and terminal $g$.

</div>
<div class="panel panel-plain"><h4>Plain English</h4>

BSDEs and semilinear PDEs are two descriptions of the **same object**. If you can solve the PDE, you have the BSDE solution evaluated along the Markov process. Conversely, if you solve the BSDE (e.g. by deep learning), you have the PDE solution at sampled points.

**Why this is useful for the repo:**

- **High dimensions.** PDE grid methods scale exponentially. BSDE + Monte Carlo scales linearly. For 3-D LOB (state $= (S, q, \text{signal})$), either works. For higher dim, BSDE wins.

- **Non-linearities.** The generator $f$ is naturally non-linear (HJB has a $\sup$ over controls). BSDEs handle this gracefully.

</div>
<div class="panel panel-code"><h4>Code</h4>

Every call to
`bsde.f_tf(t, x, y, z)` in
[solver.py](../solver.py) is evaluating

$f(t, x, y, z)$,

and the Euler step

<pre>y_next = y - f(t,x,y,z)*dt
             + z * dW</pre>

is the stochastic version of the
PDE backward step. The solver
never touches the PDE directly —
but Feynman–Kac says it is
implicitly solving one.

</div>
</div>

## 7. Connection to stochastic control: HJB

<div class="three-layer">
<div class="panel panel-math"><h4>Math</h4>

Consider the **stochastic control** problem
$$v(t, x) = \sup_{\alpha \in \mathcal{A}} \mathbb{E}\!\left[\int_t^T L(s, X_s^{\alpha}, \alpha_s)\,ds + g(X_T^{\alpha}) \;\middle|\; X_t^{\alpha} = x\right],$$

where $\alpha$ is an admissible control and
$dX^{\alpha} = b(t, x, \alpha)\,dt + \sigma(t, x)\,dW$.

The **Hamilton–Jacobi–Bellman equation** reads

$$\partial_t v + \sup_\alpha \Bigl\{ b \cdot \nabla v + L(t, x, \alpha) \Bigr\} + \tfrac{1}{2}\mathrm{Tr}\bigl(\sigma\sigma^\top \nabla^2 v\bigr) = 0.$$

Setting $f(t, x, y, z) = \sup_\alpha \{ b(t,x,\alpha) \cdot z / \sigma + L(t, x, \alpha) \}$ (or similar) turns HJB into a BSDE.

</div>
<div class="panel panel-plain"><h4>Plain English</h4>

HJB is a semilinear PDE. By §6 it is a BSDE.

The supremum inside $f$ is the **Hamiltonian** — the best-response to a given $(Y, Z)$. In market making, the control $\alpha$ is the pair $(\delta_a, \delta_b)$ of quotes, and the Hamiltonian maximises expected spread revenue.

The first-order condition on this supremum is what Module 5 will turn into explicit quote formulae.

</div>
<div class="panel panel-code"><h4>Code</h4>

In [equations/contxiong_lob.py](../equations/contxiong_lob.py)
lines 135-160, the solver uses
the FOC to get

<pre>delta_a = 1/alpha + Z^q / sigma_q
delta_b = 1/alpha - Z^q / sigma_q</pre>

These plug into `f_tf` at
line 302-348 to close the loop:
the control depends on $Z$, and
the generator depends on the
control.

</div>
</div>

## 8. Where to look in the codebase

The repo's BSDE machinery is structured in two layers:

1. An **abstract interface** [equations/base.py](../equations/base.py): every equation provides `sample`, `f_tf`, `g_tf`.
2. Concrete **equation classes** in `equations/contxiong_lob*.py` that override those methods for specific models.


<div class="three-layer">
<div class="panel panel-math"><h4>Math</h4>

**Cont–Xiong base generator** ([contxiong_lob.py:302-348](../equations/contxiong_lob.py)):

$$f(t, x, y, z) \;=\; -r\,y \;-\; \psi(q) \;+\; \lambda_a f_a(\delta_a)\delta_a \;+\; \lambda_b f_b(\delta_b)\delta_b,$$

with quotes $\delta_{a,b}$ given by the HJB FOC (Module 5).

**Terminal** ([contxiong_lob.py:367-370](../equations/contxiong_lob.py)):

$$g(T, x) \;=\; -\psi(q_T),$$

where $\psi(q) = \phi q^2$ (quadratic inventory penalty).

</div>
<div class="panel panel-plain"><h4>Plain English</h4>

Reading the generator term-by-term:

- $-r y$ — discounting (time value of money).
- $-\psi(q)$ — running inventory penalty (you pay for holding risk).
- $+ \lambda_a f_a \delta_a$ — expected revenue from ask fills per unit time.
- $+ \lambda_b f_b \delta_b$ — expected revenue from bid fills per unit time.

The terminal says: "whatever inventory you have left at $T$, you liquidate at a penalty $\phi q_T^2$."

So $Y_t$ is the *expected discounted PnL going forward*, and $Z_t$ is the *sensitivity of future PnL* to market shocks.

</div>
<div class="panel panel-code"><h4>Code</h4>

<pre>def f_tf(self, t, x, y, z):
    q = x[..., 1:2]
    Z_q = z[..., 1:2]
    da = 1/alpha + Z_q / sigma_q
    db = 1/alpha - Z_q / sigma_q
    fa = self._exec_prob_tf(da)
    fb = self._exec_prob_tf(db)
    running = (-r*y - phi*q**2
               + lam_a*fa*da
               + lam_b*fb*db)
    return running</pre>

Simplified view of
[contxiong_lob.py:302-348](../equations/contxiong_lob.py).

</div>
</div>

## 9. Numerical demo: solving a non-linear BSDE by Monte Carlo

To ground the theory, let's solve a simple non-linear BSDE exactly by Monte Carlo, without any neural networks. The generator is

$$f(t, x, y, z) = -r y + \tfrac{1}{2} z^2.$$

(This is the generator of a quadratic BSDE related to entropic risk measures — well-posed for small $T$.)

We discretise the BSDE backward: starting from $Y_T = g(X_T)$, we estimate
$$Y_{t_k} \approx \mathbb{E}\bigl[Y_{t_{k+1}} - f(t_{k+1}, X_{t_{k+1}}, Y_{t_{k+1}}, Z_{t_{k+1}})\,\Delta t \,\bigm|\, \mathcal{F}_{t_k}\bigr].$$

This conditional expectation is approximated via regression on basis functions of $X_{t_k}$ (Longstaff–Schwartz style).


In [ ]:
# Tsitsiklis-Van Roy / Longstaff-Schwartz backward induction for a toy BSDE.
# Forward: dX = mu dt + sigma dW, X_0 = 0. Terminal: g(X_T) = X_T^2.
# Generator: f(t, x, y, z) = -r y.  (linear so we can verify against Feynman-Kac.)

mu, sigma, T, N = 0.1, 0.3, 1.0, 50
n_paths, r_rate = 20_000, 0.05
dt = T / N

rng = np.random.default_rng(42)
dW = rng.standard_normal((N, n_paths)) * np.sqrt(dt)
X = np.zeros((N + 1, n_paths))
for k in range(N):
    X[k + 1] = X[k] + mu * dt + sigma * dW[k]

# Terminal
Y = np.zeros((N + 1, n_paths))
Y[-1] = X[-1] ** 2

# Backward regression on polynomial basis
def phi(x):
    return np.stack([np.ones_like(x), x, x ** 2, x ** 3], axis=1)  # (n_paths, 4)

for k in range(N - 1, -1, -1):
    # Euler update for Y (linear generator f = -r Y)
    Y_next = Y[k + 1]
    target = Y_next * (1 + r_rate * dt)  # i.e. Y_k ≈ Y_{k+1} + r Y_{k+1} dt
    # Regress target onto phi(X_k)
    basis = phi(X[k])
    coef, *_ = np.linalg.lstsq(basis, target, rcond=None)
    Y[k] = basis @ coef

Y0_bsde = Y[0].mean()
# Feynman-Kac closed form: Y_0 = exp(-r T) E[X_T^2] with linear generator +r
# (Our discretisation uses +r because f = -r Y means dY/dt = +r Y going backward,
#  so Y_t = e^{-r(T-t)} E[g(X_T)|F_t].)
ET = (mu * T) ** 2 + sigma ** 2 * T  # E[X_T^2] for arithmetic BM + drift
Y0_closed = np.exp(-r_rate * T) * ET

print(f"Toy BSDE with linear generator f(y) = -r y and g(x) = x^2:")
print(f"  Regression estimate Y_0  = {Y0_bsde:.4f}")
print(f"  Feynman-Kac closed form  = {Y0_closed:.4f}")

fig, ax = plt.subplots()
t_grid = np.linspace(0, T, N + 1)
ax.plot(t_grid, Y.mean(axis=1), lw=1.5, label="regression estimate of E[Y_t]")
ax.plot(t_grid, np.exp(-r_rate * (T - t_grid)) * ((mu * t_grid) ** 2 + sigma ** 2 * t_grid),
        lw=1.2, ls="--", label="Feynman-Kac")
ax.set_xlabel("t"); ax.set_ylabel(r"$\mathbb{E}[Y_t]$")
ax.set_title("Backward-induction BSDE solver vs closed form")
ax.legend(); plt.tight_layout(); plt.show()


<div class="callout callout-insight"><h4>From regression to neural nets</h4>

This regression approach (Tsitsiklis–Van Roy, Longstaff–Schwartz) is the BSDE numerical method of the 2000s. It works in low dimensions but suffers from basis explosion in high dimensions. **Deep BSDE** (Module 6) replaces the polynomial basis with a neural network — preserving the backward-induction idea but scaling to hundreds of dimensions. Architecturally it's the same idea: learn $Y$ and $Z$ as functions of $X_t$.

</div>

## 10. Exercises

<div class="callout callout-exercise"><h4>Exercise 1</h4>

**From BSDE to PDE.** For the forward SDE $dX = \sigma\,dW$ (Brownian motion), generator $f \equiv 0$, and terminal $g(x) = x^2$, use Feynman–Kac to write down $u(t, x)$ explicitly, then read off $Y_t$ and $Z_t$ from the identifications $Y_t = u(t, X_t)$, $Z_t = \sigma \partial_x u(t, X_t)$.

<details class="solution"><summary>Explanation</summary>

Here $Y_t = \mathbb{E}[X_T^2 \mid X_t] = X_t^2 + \sigma^2(T - t)$, so $u(t, x) = x^2 + \sigma^2 (T - t)$. Then $\partial_x u = 2x$, hence $Z_t = \sigma \cdot 2 X_t = 2\sigma X_t$. Check: the BSDE $-dY_t = -Z_t\,dW_t$ becomes $dY_t = 2\sigma X_t\,dW_t$, which by Itô on $X_t^2$ (applied with zero drift) gives exactly $d(X_t^2) = 2 X_t\,dX_t + \sigma^2\,dt = 2\sigma X_t\,dW_t + \sigma^2\,dt$, so $Y_t = X_t^2 - \sigma^2(T - t) + \text{const}$. Consistent.

</details></div>

<div class="callout callout-exercise"><h4>Exercise 2</h4>

**Identify the generator in the repo.** Open [`equations/contxiong_lob.py`](../equations/contxiong_lob.py) around lines 302-350 (the `f_tf` method). Match the code term-by-term to the mathematical generator $$f(t, x, y, z) = -r y - \phi q^2 + \lambda_a f_a \delta_a + \lambda_b f_b \delta_b.$$ Which code symbol corresponds to each mathematical symbol? Which line computes the quadratic inventory penalty $\phi q^2$?

<details class="solution"><summary>Explanation</summary>

- `self.r` ↔ $r$ (discount rate)
- `self.phi` ↔ $\phi$ (inventory penalty coefficient)
- `q = x[..., 1:2]` extracts the inventory coordinate
- `Z_q = z[..., 1:2]` extracts the inventory-adjoint component of $Z$
- `da, db` ↔ $\delta_a, \delta_b$ — computed by FOC `1/alpha +/- Z_q / sigma_q`
- `fa, fb` ↔ $f_a, f_b$ — execution probabilities
- The penalty $\phi q^2$ is computed inline as `self.phi * q**2` (or via `self._penalty_tf(q)` if a non-quadratic penalty is chosen in the config).

</details></div>

<div class="callout callout-exercise"><h4>Exercise 3</h4>

**Estimate $Y_0$ for the linear call-option BSDE above via BSDE backward induction** (instead of Monte Carlo). Use GBM for $X$, $g(x) = (x - K)^+$, and linear generator $f(y) = -r y$. Compare to Black–Scholes. Use the same polynomial regression scheme as the toy above.

<details class="solution"><summary>Explanation</summary>

Use GBM paths from Module 1's `euler_maruyama` helper (with $\mu = r, \sigma$ constant). The backward recursion is identical to the toy — only the terminal payoff changes. Expect close agreement with BS (within ~1% for 20k paths). Code in the next cell.

</details></div>

In [ ]:
# Solution - Exercise 3
S0, K, T, r_rate, sigma_bs = 100.0, 100.0, 1.0, 0.05, 0.20
N, n_paths = 100, 20_000
dt = T / N
rng = np.random.default_rng(7)
dW = rng.standard_normal((N, n_paths)) * np.sqrt(dt)
logS = np.zeros((N + 1, n_paths))
logS[0] = np.log(S0)
for k in range(N):
    logS[k + 1] = logS[k] + (r_rate - 0.5 * sigma_bs**2) * dt + sigma_bs * dW[k]
S_paths = np.exp(logS)

# Terminal payoff
Y = np.zeros((N + 1, n_paths))
Y[-1] = np.maximum(S_paths[-1] - K, 0.0)

# Polynomial regression backward
def phi(x):
    z = (x - S0) / S0
    return np.stack([np.ones_like(z), z, z**2, z**3, z**4], axis=1)

for k in range(N - 1, -1, -1):
    target = Y[k + 1] * (1 + r_rate * dt)
    basis = phi(S_paths[k])
    coef, *_ = np.linalg.lstsq(basis, target, rcond=None)
    Y[k] = basis @ coef

# Black-Scholes
from math import log, sqrt, erf
d1 = (log(S0/K) + (r_rate + 0.5*sigma_bs**2)*T) / (sigma_bs*sqrt(T))
d2 = d1 - sigma_bs*sqrt(T)
Nf = lambda x: 0.5 * (1 + erf(x / sqrt(2)))
bs = S0*Nf(d1) - K*np.exp(-r_rate*T)*Nf(d2)

print(f"BSDE backward induction Y_0 = {Y[0].mean():.4f}")
print(f"Black-Scholes               = {bs:.4f}")
print(f"Difference                  = {abs(Y[0].mean() - bs):.4f}  ({100*abs(Y[0].mean() - bs)/bs:.2f}% relative)")


<div class="callout callout-exercise"><h4>Exercise 4</h4>

**Read the terminal condition.** Find the `g_tf` method in [`equations/contxiong_lob.py`](../equations/contxiong_lob.py). What is $g(T, x)$ for the Cont–Xiong model? What is the economic interpretation of the negative sign?

<details class="solution"><summary>Explanation</summary>

`g_tf` returns $-\psi(q_T) = -\phi q_T^2$. The negative sign means: at terminal time $T$, any leftover inventory $q_T$ imposes a *negative* value (a liquidation cost). If $Y_t$ represents wealth (PnL), having inventory left over hurts — either due to adverse market impact of liquidation, or because the model wants to discourage ending with large positions. This is the mechanism that drives the market-maker to *mean-revert* inventory through the quotes.

</details></div>

## 11. Takeaways

| Concept | Role in the repo |
|---------|------------------|
| Terminal condition $g$ | `equations/*.py:g_tf` — usually a liquidation penalty |
| Generator $f$ | `equations/*.py:f_tf` — discounting, inventory penalty, spread revenue |
| Adjoint $Z$ | Learned by the solver as $\mathrm{NN}_t(X_t)$ — drives the quotes via HJB FOC |
| Martingale representation | Justifies existence of $Z$; Pardoux–Peng gives uniqueness |
| BSDE ↔ semilinear PDE | Feynman–Kac: the solver is implicitly solving a 3-D semilinear PDE |

## What's next

[Module 3](03_bsdes_with_jumps.ipynb) adds **jumps** — necessary because the repo's inventory dynamics are fundamentally driven by discrete order arrivals (a compound Poisson process), not pure diffusion. We'll see how the BSDE form extends to BSDEJ and where the `solver_cx_bsdej.py` file fits in.

<div class="module-nav">
<a href="01_brownian_motion.ipynb"><strong>← Prev</strong> Module 1: Brownian Motion</a>
<a href="03_bsdes_with_jumps.ipynb"><strong>Next →</strong> Module 3: BSDEs with Jumps</a>
</div>
